# Clone Repository and set up the Environment

In [1]:
!pwd

/content


In [2]:
!git clone https://github.com/VictoryChianumba/robust-eeg-models

Cloning into 'robust-eeg-models'...
remote: Enumerating objects: 3269, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 3269 (delta 48), reused 37 (delta 37), pack-reused 3214 (from 3)
Receiving objects: 100% (3269/3269), 832.22 MiB | 15.37 MiB/s, done.
Resolving deltas: 100% (1245/1245), done.
Updating files: 100% (2211/2211), done.
Filtering content: 100% (2/2), 663.03 MiB | 30.56 MiB/s, done.


In [1]:
%cd robust-eeg-models

/content/robust-eeg-models


In [3]:
!git config --global user.email "chianumbav@gmial.com"
!git config --global user.name "Victory Chianumba"

In [ ]:
# 1️⃣ Upgrade the package manager
!pip install --upgrade --quiet pip

!pip install torch torchvision torchaudio
!pip install mne moabb
!pip install torch-lr-finder
!pip install optuna
!pip install torchattacks
!pip install advertorch
!pip install foolbox
!pip install captum

# Clean uninstall of briandecode
!pip uninstall -y braindecode

# Install latest braindecode and autoattack code from GitHub (which includes braindecode CTNet)
!pip install git+https://github.com/braindecode/braindecode.git@master --no-cache-dir
!pip install git+https://github.com/fra31/auto-attack

In [4]:
import braindecode
print("Braindecode version:", braindecode.__version__)

from braindecode.models import CTNet
print("CTNet is available ✅")


Braindecode version: 1.2.0
CTNet is available ✅


In [5]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # or ":16:8" if memory is tight
os.environ["PYTHONHASHSEED"] = "0"                  # optional, extra stability


In [6]:
import torch, torchattacks, foolbox, optuna, autoattack, captum
# import advertorch
import importlib, sys, pickle, json, random, time, datetime, numbers, hashlib, subprocess

import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [10]:
from datetime import datetime
from collections import defaultdict

from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR, CosineAnnealingWarmRestarts

from braindecode import EEGClassifier
from braindecode.models import EEGNetv4, Deep4Net, CTNet
from braindecode.datasets import MOABBDataset
from braindecode.augmentation import FrequencyShift, GaussianNoise, AugmentedDataLoader, Compose, SmoothTimeMask, Mixup
from braindecode.training import mixup_criterion

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from skorch.helper import predefined_split
from skorch.callbacks import LRScheduler, EarlyStopping, Checkpoint
from sklearn.metrics import jaccard_score
from sklearn.metrics.pairwise import cosine_similarity

# Use to visualise the embeddings, COULD be used to visualise the explanations
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from scipy.stats import spearmanr

#EEGMamba-MOE approximation
from models.eeg_mamba_fft import create_eegmamba, EEGMamba

# Loading data for training

In [11]:
#import numpy as np
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
from braindecode.preprocessing import create_windows_from_events
from sklearn.model_selection import train_test_split
from skorch.helper import SliceDataset
from torch.utils.data import Subset

device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = MOABBDataset(
    dataset_name='BNCI2014001', subject_ids=[1]
)

#----------------------------------------------------------------------
# After loading we preprocess

low_cut_hz = 4.0  # low cut frequency for filtering
high_cut_hz = 38.0  # high cut frequency for filtering
# Parameters for exponential moving standardization
factor_new = 1e-3
init_block_size = 1000

preprocessors = [
    Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
    Preprocessor(
        lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
        factor=1e6,
    ),
    Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
    Preprocessor(
        exponential_moving_standardize,  # Exponential moving standardization
        factor_new=factor_new,
        init_block_size=init_block_size,
    ),
]

# Preprocess the data
preprocess(dataset, preprocessors, n_jobs=-1)

#-----------------------------------------------------------------------

trial_start_offset_seconds = -0.5
# Extract sampling frequency, check that they are same in all datasets
sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
# Calculate the window start offset in samples.
trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

# Create windows using braindecode function for this. It needs parameters to
# define how windows should be used.
windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples = int(-0.5 * sfreq) , # -0.5s before cue
    trial_stop_offset_samples = 0,   # 4.0s after cue (t=2s to t=6s)
    preload=True,
    # verbose=0
)


# ----------------------------------------------------------------------
# Split into train and test
splitted = windows_dataset.split("session")
train_set = splitted["0train"]  # Session train
test_set = splitted["1test"]  # Session evaluation

# ----------------------------------------------------------------------
# Split into train, val subsets

X_train = SliceDataset(train_set, idx=0)

y_train = np.array([y for y in SliceDataset(train_set, idx=1)])

X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T)
y_test = np.array(test_set.get_metadata().target)                   # (N,)

train_indices, val_indices = train_test_split(
      X_train.indices_, test_size=0.2, shuffle=False
  )
train_subset = Subset(train_set, train_indices)
val_subset = Subset(train_set, val_indices)

# Build simple tensors to compute stats on train windows only
X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T
train_mean = X_train.mean(axis=(0,2), keepdims=True)  # (1,C,1)
train_std  = X_train.std(axis=(0,2), keepdims=True) + 1e-6

# Optionally, keep empirical bounds for later clipping in attack
train_min = X_train.min(axis=(0,2), keepdims=True)
train_max = X_train.max(axis=(0,2), keepdims=True)


# 3) materialize test tensors (NOT SliceDataset)
X = torch.tensor(X_test, dtype=torch.float32, device=device)
y = torch.tensor(y_test, dtype=torch.long, device=device)

x, y , meta = train_set[0]
print(type(x), x.shape, x.mean(), x.std())

# print(train_mean.shape)
# print(train_std.shape)
# print(train_min.shape)
# print(train_max.shape)
# print(train_mean)
# print(train_std)
# print(train_min)
# print(train_max)

# Save these via _save_run(... train_mean=train_mean, train_std=train_std, train_min=train_min, train_max=train_max)


/usr/local/lib/python3.12/dist-packages/moabb/datasets/download.py:56: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))


MNE_DATA is not already configured. It will be set to default location in the home directory - /root/mne_data
All datasets will be downloaded to this location, if anything is already downloaded, please move manually to this location


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 42.8M/42.8M [00:00<00:00, 62.6GB/s]
SHA256 hash of downloaded file: 054f02e70cf9c4ada1517e9b9864f45407939c1062c6793516585c6f511d0325
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 43.8M/43.8M [

Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
<class 'numpy.ndarray'> (22, 1125) -0.002018633 1.0159434


Check if this loads as train_set = x, y or x, y , meta

---
Useful for later

In [11]:
sample = train_set[0]
print(type(sample))
print(len(sample))   # see what fields exist ()


<class 'tuple'>
3


## Set loading functions

In [12]:
def load_subject_data_cached(dataset, subject_id):
    cache_file = f'cache/subject_{subject_id}_processed.pkl'

    if os.path.exists(cache_file):
        # Load from cache - instant!
        with open(cache_file, 'rb') as f:
            return pickle.load(f)

    # Process and cache
    train_set, test_set, train_subset, val_subset = load_subject_data(dataset,subject_id)

    os.makedirs('cache', exist_ok=True)
    with open(cache_file, 'wb') as f:
        pickle.dump((train_set, test_set, train_subset, val_subset), f)


    return train_set, test_set, train_subset, val_subset

def load_subject_data(dataset, subject_id):

    import numpy as np
    from braindecode.datasets import MOABBDataset
    from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
    from braindecode.preprocessing import create_windows_from_events
    from sklearn.model_selection import train_test_split
    from skorch.helper import SliceDataset
    from torch.utils.data import Subset

    dataset = MOABBDataset(
        dataset_name=dataset, subject_ids=[subject_id]
    )

    #----------------------------------------------------------------------
    # After loading we preprocess

    low_cut_hz = 4.0  # low cut frequency for filtering
    high_cut_hz = 38.0  # high cut frequency for filtering
    # Parameters for exponential moving standardization
    factor_new = 1e-3
    init_block_size = 750

    preprocessors = [
        Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
        Preprocessor(
            lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
            factor=1e6,
        ),
        Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
        Preprocessor(
            exponential_moving_standardize,  # Exponential moving standardization
            factor_new=factor_new,
            init_block_size=init_block_size,
        ),
    ]

    # Preprocess the data
    preprocess(dataset, preprocessors, n_jobs=-1)

    #-----------------------------------------------------------------------

    trial_start_offset_seconds = -0.5
    # Extract sampling frequency, check that they are same in all datasets
    sfreq = dataset.datasets[0].raw.info["sfreq"]
    assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
    # Calculate the window start offset in samples.
    trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

    # Create windows using braindecode function for this. It needs parameters to
    # define how windows should be used.
    windows_dataset = create_windows_from_events(
        dataset,
        trial_start_offset_samples=trial_start_offset_samples,
        trial_stop_offset_samples=0,
        preload=True,
        # verbose=0
    )

    # ----------------------------------------------------------------------
    # Split into train and test
    splitted = windows_dataset.split("session")
    train_set = splitted["0train"]  # Session train
    test_set = splitted["1test"]  # Session evaluation

    # ----------------------------------------------------------------------
    # Split into train, val subsets

    X_train = SliceDataset(train_set, idx=0)
    y_train = np.array([y for y in SliceDataset(train_set, idx=1)])
    train_indices, val_indices = train_test_split(
        X_train.indices_, test_size=0.2, shuffle=False
    )
    train_subset = Subset(train_set, train_indices)
    val_subset = Subset(train_set, val_indices)

    return train_set, test_set, train_subset, val_subset


## Data loading sanity check

In [13]:
# Run once or sanity check
subject_id = 1
train_set, test_set, train_subset, val_subset = load_subject_data_cached("BNCI2014001", subject_id)
print(val_subset.indices)

/usr/local/lib/python3.12/dist-packages/braindecode/preprocessing/preprocess.py:71: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
[230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247
 24

## Load and inspect model

In [14]:
from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

# Extract number of chans and time steps from dataset
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = CTNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    # n_layers = 2,
    # n_experts = 8

)

# model.enable_moe(True)

# Send model to GPU
if cuda:
    model.cuda()

/usr/local/lib/python3.12/dist-packages/braindecode/util.py:52: UserWarning: torch.backends.cudnn.benchmark was set to True which may results in lack of reproducibility. In some cases to ensure reproducibility you may need to set torch.backends.cudnn.benchmark to False.
  warn(


# Training

## Set model hyper params

In [17]:

eegnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

deepconvnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

CTNet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW}
alt_params = {'lr': 1e-3, 'batch_size': 128, 'weight_decay': 5e-3, 'optimizer': torch.optim.AdamW}

mamba_params = {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW}
mamba_params1 = {'lr': 0.0016, 'batch_size': 128, 'weight_decay': 5e-4, 'optimizer': torch.optim.AdamW}

n_epochs = 500

## Single run mode

Single training run for model debugging, sanity checks and testing non braindecode architectures (EEGMammba)



In [ ]:
def make_scheduler(optimizer, last_epoch=-1):
    return CosineAnnealingWarmRestarts(optimizer, T_0=300, T_mult=1, eta_min=1e-6, last_epoch=last_epoch)

device = "cuda" if torch.cuda.is_available() else "cpu"


# Load tehe training data
train_set, test_set, train_subset, val_subset= load_subject_data_cached("BNCI2014001", 1)

# Build transforms list
transforms = [
    FrequencyShift(probability=0.3, sfreq=250, max_delta_freq=0.3),
    GaussianNoise(probability=0.3, std=0.0),
    # Mixup(alpha=0.2,  beta_per_sample=True),               # ← returns (x, (y1, y2, lam))
]


# Extract model params from dataset, initialise model and set hyper-parameters
classes = torch.unique(torch.tensor([sample[1] for sample in train_subset])).tolist()
n_classes = len(classes)
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]

model = EEGMamba(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,


)

# Hyper params
params = mamba_params

# Toggle this to enable mamba (For EEGMamba only)
# model.enable_moe(True)

# Enable MoE head instead of standard classifier (For EEGMamba only)
# model.use_moe = False

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def make_scheduler(optimizer, last_epoch=-1):
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=10, last_epoch=last_epoch)
    cosine = CosineAnnealingLR(optimizer, T_max=n_epochs-10, last_epoch=last_epoch)
    return CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

# Create new classifier with best parameters
clf = EEGClassifier(
    model,
    # iterator_train=AugmentedDataLoader,
    # iterator_train__transforms=transforms,
    # iterator_train__shuffle=True,

    # dataset = aug_train,
    criterion=torch.nn.CrossEntropyLoss,
    # criterion__base_criterion=torch.nn.CrossEntropyLoss(reduction='none'),

    train_split=predefined_split(val_subset),  # Use all training data

    optimizer=params['optimizer'],

    # Looking at subjects
    # optimizer = torch.optim.SGD,
    # optimizer__momentum=0.9,

    optimizer__lr=params['lr'],
    optimizer__weight_decay=params['weight_decay'],
    batch_size=params['batch_size'],
    callbacks=[
        "accuracy",
        # ("lr_scheduler", LRScheduler(make_scheduler))
        # ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
        # ("early_stopping", EarlyStopping(patience=200, monitor="valid_acc")),
    ],
    device=device,
    classes=classes,
    max_epochs=500,
)

# Train on full training set
clf.fit(train_subset, y=None)

# Evaluate the model after training
y_test = test_set.get_metadata().target
test_acc = clf.score(test_set, y=y_test)

print(f"Val acc with MoE: {(test_acc * 100):.2f}%")

import numbers


# Record Baselines for chosen models

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)  # PyTorch 1.11+
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_environment_fingerprint():
    pip_freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"]).decode()
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    return {
        "python": sys.version,
        "pytorch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cudnn_version": torch.backends.cudnn.version(),
        "pip_freeze": pip_freeze.splitlines(),
    }

def tiny_json(base, model, id, seed, skorch_params, backend, notes):

  os.makedirs(base, exist_ok=True)
  tiny = {
      "model_name": model,
      "subject_id": id,
      "seed": seed,
      "bandpass": {"l_freq": 4.0, "h_freq": 38.0},
      "unit_scale_to_uV": 1e6,
      "ems": {"factor_new": 1e-3, "init_block_size": 750},
      "trial_start_offset_seconds": -0.5,
      "windowing": "create_windows_from_events(session split: 0train/1test)",
      "zscore_applied": False,
      "skorch_params": skorch_params,
      "backend":backend,
      "notes": notes
  }
  return tiny

def safe_model_config(model_config: dict) -> dict:
    """Convert model_config into a JSON-serializable dict."""
    safe_cfg = {}
    for k, v in model_config.items():
        if k == "model_class":
            # store full module path + class name
            safe_cfg[k] = f"{v.__module__}.{v.__name__}" if hasattr(v, "__module__") else str(v)
        elif k == "training":
            safe_training = {}
            for tk, tv in v.items():
                if tk == "optimizer":
                    # also store optimizer class name
                    safe_training[tk] = f"{tv.__module__}.{tv.__name__}" if hasattr(tv, "__module__") else str(tv)
                else:
                    # assume JSON-friendly scalar
                    safe_training[tk] = tv
            safe_cfg[k] = safe_training
        else:
            safe_cfg[k] = v if isinstance(v, (int, float, str, bool, type(None))) else str(v)
    return safe_cfg


# ==============================================================================

def train_single_run(model_name, subject_id, seed, dataset):

    # 0. RNG reproducibility ---------------------------------------------------

    set_all_seeds(seed)
    rng_state_np    = np.random.get_state()
    rng_state_torch = torch.get_rng_state()
    env_fp = get_environment_fingerprint()

    print(f"\n=== Processing Subject {subject_id} for seed {seed} ===")

    # 1. data ------------------------------------------------------------------
    # Load data
    train_set, test_set, train_subset, val_subset = load_subject_data_cached(dataset, subject_id)

    # Build simple tensors to compute stats on train windows only
    X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T)
    train_mean = X_train.mean(axis=(0,2), keepdims=True)  # (1,C,1)
    train_std  = X_train.std(axis=(0,2), keepdims=True) + 1e-6

    # Empirical bounds for later clipping in attack
    train_min = X_train.min(axis=(0,2), keepdims=True)
    train_max = X_train.max(axis=(0,2), keepdims=True)

    # expose indices explicitly
    train_idx = train_subset.indices
    val_idx   = val_subset.indices
    test_idx  = np.arange(len(test_set))

    # 2. model & config --------------------------------------------------------

    # Get model config
    config = MODEL_CONFIGS[model_name]

    # Extract model params from dataset, initialise model and set hyper-parameters
    # classes needed for clf
    classes = torch.unique(torch.tensor([sample[1] for sample in train_subset])).tolist()
    n_classes = len(classes)
    n_channels = train_subset[0][0].shape[0]
    n_times = train_subset[0][0].shape[1]

    model = config['model_class'](
        n_chans=n_channels,
        n_outputs=n_classes,
        n_times=n_times,
    )

    # Special handling for EEGMamba
    if model_name == 'EEGMamba':
        model.enable_moe(False)  # Use standard classifier for baseline. Paper explicitly removes moe modules for
                                # single use mamba
        # Enable MoE head instead of standard classifier (For multi-use only)
        model.use_moe = False

    # 4. fit -------------------------------------------------------------------

    # Create new classifier with best parameters
    clf = EEGClassifier(
        model,
        criterion=torch.nn.CrossEntropyLoss,
        train_split=predefined_split(val_subset),  # Use all training data
        optimizer=config['training']['optimizer'],
        optimizer__lr=config['training']['lr'],
        optimizer__weight_decay=config['training']['weight_decay'],
        batch_size=config['training']['batch_size'],
        callbacks=["accuracy"],
        device=device,
        classes=classes,
        max_epochs=500,
    )

    # Train on full training set
    clf.fit(train_subset, y=None)

    # 5. test accuracy ---------------------------------------------------------

    # Evaluate the model after training
    y_test = test_set.get_metadata().target
    test_accuracy = clf.score(test_set, y = y_test)

    # 6. save everything -------------------------------------------------------

    _save_run(model_name, subject_id, seed,
              clf, test_set, rng_state_np, rng_state_torch,
              train_idx, val_idx, test_idx,
              train_mean, train_std, train_min, train_max,
              config, env_fp)

    return test_accuracy

# ==============================================================================

# Generate final baseline table
def create_baseline_table(results):
    """Create a nice table of baselines"""
    rows = []

    for model_name in results.keys():
        for subject_id in subjects:
            scores = results[model_name][subject_id]
            valid_scores = [s for s in scores if not np.isnan(s)]

            if valid_scores:
                mean_acc = np.mean(valid_scores)
                std_acc = np.std(valid_scores)
                n_valid = len(valid_scores)
            else:
                mean_acc = std_acc = n_valid = np.nan

            rows.append({
                'Model': model_name,
                'Subject': subject_id,
                'Mean_Accuracy': mean_acc,
                'Std_Accuracy': std_acc,
                'N_Valid_Runs': n_valid,
                'Individual_Scores': scores
            })

    return pd.DataFrame(rows)


# ==============================================================================



def _save_run(model_name, subject_id, seed, clf, test_set,
              rng_state_np, rng_state_torch,
              train_idx, val_idx, test_idx,
              train_mean=None, train_std=None,
              train_min=None, train_max=None,
              model_config: dict = None,
              env_fingerprint: dict = None,
              device: str=device):
    """
    clf          : fitted skorch net (clf.module_ is torch.nn.Module)
    test_set     : braindecode Dataset (getitem -> (x, y))
    *_idx        : np.ndarray of ints
    train_mean/std/min/max : arrays shaped (C,) or (1,C,1) (we'll serialize as lists)
    model_config : dict of arch + training hyperparams
    env_fingerprint : dict from get_environment_fingerprint()
    """

    base = f"{SAVE_DIR}/{model_name}/{model_name}_S{subject_id}_seed{seed}"
    os.makedirs(base, exist_ok=True)

    # ---- 0) Metadata header ----
    meta = {
        "model_name": model_name,
        "subject_id": int(subject_id),
        "seed": int(seed),
    }

    if model_config is not None:
        meta["model_config"] = safe_model_config(model_config)
    if env_fingerprint is not None:
        meta["environment"] = env_fingerprint
    json.dump(meta, open(f"{base}/meta.json", "w"), indent=2)

    # ---- 1) Checkpoint (state_dict + optimizer) ----
    torch.save({
        "state_dict": clf.module_.state_dict(),
        "optimizer": getattr(clf, "optimizer_", None).state_dict() if hasattr(clf, "optimizer_") else None,
        "seed": seed
    }, f"{base}/checkpoint.pth")

    # ---- 2) Training curves/history ----
    hist = clf.history_
    # Adjust keys if needed:
    train_acc = [e.get("train_accuracy", e.get("train_acc")) for e in hist]
    val_acc   = [e.get("valid_accuracy", e.get("val_acc")) for e in hist]
    train_loss= [e.get("train_loss") for e in hist]
    val_loss  = [e.get("valid_loss", e.get("val_loss")) for e in hist]
    curves = {"train_acc": train_acc, "val_acc": val_acc,
              "train_loss": train_loss, "val_loss": val_loss}
    json.dump(curves, open(f"{base}/curves.json", "w"), indent=2)

    # ---- 3) Test logits (CLEAN) + loss vector ----
    # Build X_test, y_test from the Dataset (no shuffling!)
    X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T)
    y_test = np.array(test_set.get_metadata().target)                  # (N,)
    clf.module_.eval().to(device)
    with torch.no_grad():
        X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
        logits_t = clf.infer(X_test_t)   # shape (N, num_classes)
        logits = logits_t.detach().cpu().numpy()
    np.save(f"{base}/test_logits_clean.npy", logits)
    np.save(f"{base}/y_test.npy", y_test)

    loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
    y_test_t = torch.tensor(y_test, device=device, dtype=torch.long)
    logits_ten = torch.tensor(logits, device=device, dtype=torch.float32)
    loss_vec = loss_fn(logits_ten, y_test_t).cpu().numpy()
    np.save(f"{base}/test_loss_vector.npy", loss_vec)

    # ---- 4) RNG states ----
    pickle.dump({"numpy": rng_state_np, "torch": rng_state_torch}, open(f"{base}/rng_state.pkl", "wb"))

    # ---- 5) Splits ----
    splits = {"train_idx": train_idx.tolist(),
              "val_idx":   val_idx.tolist(),
              "test_idx":  test_idx.tolist()}
    json.dump(splits, open(f"{base}/splits.json", "w"), indent=2)

    # ---- 6) Preprocessing statistics (per-channel) ----
    prep = {"zscore_applied": False}
    if train_mean is not None:
        prep["train_mean"] = np.array(train_mean).reshape(-1).tolist()
    if train_std is not None:
        prep["train_std"]  = np.array(train_std).reshape(-1).tolist()
    if train_min is not None:
        prep["train_min"]  = np.array(train_min).reshape(-1).tolist()
    if train_max is not None:
        prep["train_max"]  = np.array(train_max).reshape(-1).tolist()
    json.dump(prep, open(f"{base}/preprocessing.json", "w"), indent=2)

    # ---- 7) Attack metadata (placeholder file to append later) ----
    # You will fill this AFTER you run attacks; we create an empty schema now for consistency.
    attack_meta = {
        "whitebox": {},
        "blackbox": {}
    }
    json.dump(attack_meta, open(f"{base}/attack_metadata.json", "w"), indent=2)

    # ---- 8) README for the run folder ----
    with open(f"{base}/README.txt", "w") as f:
        f.write(
            "Artifacts:\n"
            "- checkpoint.pth: model+optimizer state_dict\n"
            "- curves.json: train/val accuracy/loss per epoch\n"
            "- test_logits_clean.npy: logits on test set (clean)\n"
            "- test_loss_vector.npy: per-sample CE loss on test set (clean)\n"
            "- y_test.npy: test labels\n"
            "- rng_state.pkl: RNG snapshots (numpy/torch)\n"
            "- splits.json: train/val/test indices (no leakage)\n"
            "- preprocessing.json: channelwise stats\n"
            "- attack_metadata.json: to be populated after attacks\n"
            "- meta.json: model/subject/seed, environment fingerprint\n"
        )

    # ---------------------------
    # Tiny JSON: per-run manifest
    # ---------------------------
    # grab skorch hyperparams (safe dict)
    try:
        skorch_params = {}
        for k, v in clf.get_params().items():
            if isinstance(v, (str, bool)):
                skorch_params[k] = v
            elif isinstance(v, numbers.Number):
                skorch_params[k] = float(v) if isinstance(v, float) else int(v)
    except Exception:
        skorch_params = {}

    # attach environment + backend determinism fingerprint
    backend = {
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda if hasattr(torch.version, "cuda") else None,
        "cudnn_version": torch.backends.cudnn.version(),
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
    }

    # small cache manifest (see function below)

    tiny = tiny_json(
        base, model_name, subject_id, seed, skorch_params, backend,
        notes="baseline training run"
    )

    with open(f"{base}/tiny.json", "w") as f:
        json.dump(tiny, f, indent=2)


# ----------------------------------------------------------------------------------------------------------------
# Baseline Run
# ----------------------------------------------------------------------------------------------------------------

seeds = [42, 123, 2024, 31415, 999]   # any 3–5 different seeds

datasets = {
    "BNCIv2": ("BNCI2014001", 9),
    "DEAP": 32
}

dataset, n_subjects = datasets["BNCIv2"]
subjects = list(range(1, n_subjects+1))


SAVE_DIR = "results"          # change if you want
os.makedirs(SAVE_DIR, exist_ok=True)

# Define your models and their hyperparameters
MODEL_CONFIGS = {
    'EEGNet': {
        'model_class': EEGNetv4,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'DeepConvNet': {
        'model_class': Deep4Net,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'CTNet': {
        'model_class': CTNet,
        'training':  {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW, 'scheduler': False}
    },
    'EEGMamba': {
        'model_class': EEGMamba,
        'training': {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW, 'scheduler': False}
    }
}

# Store all results: results[model_name][subject_id] = [acc1, acc2, acc3, acc4, acc5]
all_results = defaultdict(lambda: defaultdict(list))


# Main loop
"""
Uncomment Main loop to run and save baslines
"""

# for model_name in MODEL_CONFIGS.keys():
#     print(f"\n{'='*60}")
#     print(f"RUNNING BASELINE FOR {model_name.upper()}")
#     print(f"{'='*60}")

#     for subject_id in subjects:
#         print(f"\n--- Subject {subject_id} ---")

#         subject_scores = []
#         for seed in seeds:
#             print(f"  Seed {seed}: RUNNING")
#             try:
#                 accuracy = train_single_run(model_name, subject_id, seed, dataset)
#                 subject_scores.append(accuracy)
#                 print(f"  Seed {seed}: {accuracy:.4f}")
#             except Exception as e:
#                 print(f"  Seed {seed}: FAILED ({e})")
#                 subject_scores.append(np.nan)

#         # Store results for this (model, subject) pair
#         all_results[model_name][subject_id] = subject_scores

#         # Calculate stats for this subject
#         valid_scores = [s for s in subject_scores if not np.isnan(s)]
#         if valid_scores:
#             mean_acc = np.mean(valid_scores)
#             std_acc = np.std(valid_scores)
#             print(f"  Subject {subject_id} baseline: {mean_acc:.4f} ± {std_acc:.4f}")
#         else:
#             print(f"  Subject {subject_id}: ALL RUNS FAILED")

# # Create and display results
# baseline_df = create_baseline_table(all_results)
# print(f"\n{'='*80}")
# print("FINAL BASELINE RESULTS")
# print(f"{'='*80}")

# # Subject-wise baselines
# for model_name in MODEL_CONFIGS.keys():
#     print(f"\n{model_name}:")
#     model_data = baseline_df[baseline_df['Model'] == model_name]

#     subject_means = []
#     for _, row in model_data.iterrows():
#         if not np.isnan(row['Mean_Accuracy']):
#             print(f"  Subject {row['Subject']}: {row['Mean_Accuracy']:.4f} ± {row['Std_Accuracy']:.4f}")
#             subject_means.append(row['Mean_Accuracy'])
#         else:
#             print(f"  Subject {row['Subject']}: FAILED")

#     # Dataset-wide average
#     if subject_means:
#         dataset_mean = np.mean(subject_means)
#         dataset_std = np.std(subject_means)
#         print(f"  → Dataset average: {dataset_mean:.4f} ± {dataset_std:.4f}")
#     else:
#         print(f"  → Dataset average: FAILED")

# # Save results
# baseline_df.to_csv('baseline_results.csv', index=False)
# print(f"\nResults saved to baseline_results.csv")

# Adversarial attacks


## Initialize model and load checkpoints

In [15]:

from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

# Extract number of chans and time steps from dataset
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = CTNet(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    # n_layers = 2,
    # n_experts = 8

)

# model.enable_moe(True)

# Send model to GPU
if cuda:
    model.cuda()

# Print original CTNet keys
# print(model.state_dict().keys())
path = torch.load('/content/robust-eeg-models/results/CTNet/CTNet_S1_seed123/checkpoint.pth')
print(path.keys())
model.load_state_dict(path['state_dict'])   # <- no .eval() here
model.eval()


x, y , meta = train_set[0]
print(type(x), x.shape, x.mean(), x.std())


dict_keys(['state_dict', 'optimizer', 'seed'])
<class 'numpy.ndarray'> (22, 1125) 0.002506595 1.0190344


## Initialise variables

In [18]:
eps_grid = [0.01, 0.02, 0.03, 0.05]
batch_size = 128
steps = 40
alpha = eps/8
restarts=5

## Set up, helper methods for adversarial attacks and eval mode

In [ ]:
import numpy as np, torch, json
from sklearn.metrics import auc

import torch, numpy as np, json
import torch.nn.functional as F
import torchattacks as ta
import foolbox as fb
from autoattack import AutoAttack
from scipy.stats import spearmanr
from sklearn.metrics import jaccard_score
from sklearn.metrics.pairwise import cosine_similarity
import captum.attr as CA
from sklearn.metrics import confusion_matrix
import numpy as np


device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()

# Expect: X (N,C,T) float, y (N,) long, and stats train_min, train_max, train_std
assert X.dim()==3 and y.dim()==1, "X should be (N,C,T), y should be (N,)"
X = X.to(device).float().requires_grad_(False)

################################################################
y = y.to(device).long() #####NOTE THIS CAUSES AN ERROR
###############################################################


train_min_t = torch.tensor(train_min, dtype=torch.float32, device=device)  # (1,C,1) okay
train_max_t = torch.tensor(train_max, dtype=torch.float32, device=device)
train_std_np = np.array(train_std).reshape(-1)  # (C,)

def per_channel_clamp(x, vmin, vmax): return torch.max(torch.min(x, vmax), vmin)

# Signal to noise ratio
def snr_db(x, x_adv):
    d = x_adv - x
    num = x.pow(2).sum((1,2)).sqrt()
    den = d.pow(2).sum((1,2)).sqrt().clamp_min(1e-12)
    return (20.0 * torch.log10(num/den)).detach().cpu().numpy()

def eps_to_uV_per_channel(eps, train_std_1d): return (eps*train_std_1d).tolist()

# Interpretability helpers (E_clean/E_adv: (N,C,T))
def ch_scores(E): return E.abs().mean(dim=-1).cpu().numpy()      # (N,C)

def spearman_ch(Ec, Ea):
    """ Uses clean explanations and adversarial explanations for spearman score """
    sc, sa = ch_scores(Ec), ch_scores(Ea)
    r = [spearmanr(sc[i], sa[i]).statistic for i in range(sc.shape[0])]
    return float(np.nanmean(r))

def jaccard_topk(Ec, Ea, k=5):
    sc, sa = ch_scores(Ec), ch_scores(Ea)
    J = []
    for i in range(sc.shape[0]):
        kc = np.argsort(-sc[i])[:k]; ka = np.argsort(-sa[i])[:k]
        mask_c = np.isin(np.arange(sc.shape[1]), kc).astype(int)
        mask_a = np.isin(np.arange(sc.shape[1]), ka).astype(int)
        J.append(jaccard_score(mask_c, mask_a, average="binary"))
    return float(np.mean(J))

def cosine_maps(Ec, Ea):
    v1 = Ec.flatten(1).cpu().numpy(); v2 = Ea.flatten(1).cpu().numpy()
    return float(np.mean([cosine_similarity(v1[i:i+1], v2[i:i+1])[0,0] for i in range(len(v1))])) #nanmean????

def norm_l2_shift(Ec, Ea):
    num = (Ea - Ec).flatten(1).norm(dim=1)
    den = Ec.flatten(1).norm(dim=1) + 1e-8
    return float((num/den).mean().item())

def robust_auc(eps_list, acc_list):
    """ Measuring the Robust-accuracy over ε"""
    # eps_list, acc_list: same length; AUC via trapezoid
    return float(auc(eps_list, acc_list))

def per_class_acc(y_true, y_pred, n_classes=None):
    if n_classes is None:
        n_classes = int(max(y_true.max(), y_pred.max()) + 1)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
    with np.errstate(divide='ignore', invalid='ignore'):
        accs = np.diag(cm) / cm.sum(axis=1)
    return np.nan_to_num(accs)  # shape (n_classes,)

# Example usage inside your per-ε loop (after you have logits_adv and y_test):
# y_pred_adv = logits_adv.argmax(1)
# per_class = per_class_acc(y_test, y_pred_adv, n_classes=int(y_test.max()+1))
# results['per_eps'][-1]['per_class_acc'] = per_class.tolist()

rng = np.random.default_rng(123)

def bootstrap_ci_acc(y_true, y_pred, B=2000, alpha=0.05):
    N = len(y_true)
    hits = (y_true == y_pred).astype(np.float32)
    boots = []
    for _ in range(B):
        idx = rng.integers(0, N, N)
        boots.append(hits[idx].mean())
    lo = np.percentile(boots, 100*alpha/2)
    hi = np.percentile(boots, 100*(1-alpha/2))
    return float(lo), float(hi)

# Example usage inside your per-ε loop:
# lo, hi = bootstrap_ci_acc(y_test, y_pred_adv, B=1000)
# results['per_eps'][-1]['acc_adv_CI95'] = [lo, hi]


import json, matplotlib.pyplot as plt

def save_epsilon_curve(results, out_json, out_png):
    # assumes results has keys: 'eps_grid' and per_eps[i]['acc_adv'] + ['acc_adv_CI95']
    curve = {
        "eps": results["eps_grid"],
        "acc_adv": [pe["acc_adv"] for pe in results["per_eps"]],
        "acc_adv_CI95": [pe.get("acc_adv_CI95", None) for pe in results["per_eps"]],
    }
    json.dump(curve, open(out_json, "w"), indent=2)

    # quick plot
    eps = np.array(curve["eps"], float)
    acc = np.array(curve["acc_adv"], float)
    plt.figure()
    plt.plot(eps, acc, marker='o')
    # optional CI band if present
    if all(ci is not None for ci in curve["acc_adv_CI95"]):
        lo = np.array([ci[0] for ci in curve["acc_adv_CI95"]], float)
        hi = np.array([ci[1] for ci in curve["acc_adv_CI95"]], float)
        plt.fill_between(eps, lo, hi, alpha=0.2)
    plt.xlabel("ε (L∞)")
    plt.ylabel("Robust accuracy")
    plt.title("ε–curve (robust accuracy vs ε)")
    plt.grid(True, alpha=0.3)
    plt.savefig(out_png, bbox_inches='tight', dpi=150)
    plt.close()

# After you finish the ε sweep:
# save_epsilon_curve(results,
#                    out_json=f"{run_dir}/epsilon_curve.json",
#                    out_png =f"{run_dir}/epsilon_curve.png")



## Set clean baselines

In [ ]:
with torch.no_grad():
    logits_clean = model(X)
clean_acc = float((logits_clean.argmax(1) == y).float().mean().item())

# Explanations (Integrated Gradients; LRP optional if you verified rules)
ig = CA.IntegratedGradients(model)
E_clean = ig.attribute(X, target=y, n_steps=50)    # (N,C,T), float32


## White‑box L∞ attacks (FGSM, BIM, PGD, MIM)

In [ ]:
def run_iter_attack(atk_ctor, atk_name, eps_grid, steps=None, alpha_rule=lambda e: e/8):

    """
    atk_ctor: The attack constructor for this iterated attack
    atk_name: Attack name
    eps_grid: List of epsilon values
    steps: number of steps
    alpha_rule: alpha rule function (e.g., lambda e: e/8)
    """

    out = {"attack": atk_name, "eps_grid": eps_grid, "rows": []}
    N = X.size(0)
    for eps in eps_grid:
        atk = atk_ctor(eps) if steps is None else atk_ctor(eps, steps, alpha_rule(eps))
        preds_adv, l2_succ, snrs = [], [], []
        # batch to control memory
        B = 128
        X_adv_all = []
        for i in range(0, N, B):
            Xi, yi = X[i:i+B], y[i:i+B]
            Xi = Xi.detach().clone().requires_grad_(True)
            x_adv = atk(Xi, yi).detach()
            x_adv = per_channel_clamp(x_adv, train_min_t, train_max_t)
            X_adv_all.append(x_adv)
            with torch.no_grad():
                pa = model(x_adv).argmax(1)
            preds_adv.append(pa.cpu())
            delta = (x_adv - Xi).flatten(1)
            l2 = delta.norm(p=2, dim=1).cpu().numpy()
            succ = (pa != yi).cpu().numpy()
            l2_succ.extend(l2[succ])
            snrs.extend(snr_db(Xi, x_adv))
        preds_adv = torch.cat(preds_adv)
        adv_acc = float((preds_adv == y.cpu()).float().mean().item())
        asr = float((preds_adv != y.cpu()).float().mean().item())# = adv_acc (untargeted)
        med_l2 = float(np.median(l2_succ)) if len(l2_succ) else float("nan")

        X_adv = torch.cat(X_adv_all, 0)
        # explanations on a subsample if needed (speed): here do all for clarity
        E_adv = ig.attribute(X_adv, target=y, n_steps=50)

        row = {
            "eps": float(eps),
            "eps_uV_per_channel": eps_to_uV_per_channel(eps, train_std_np),
            "clean_acc": clean_acc,
            "adv_acc": adv_acc,
            "asr": asr,
            "median_L2_success": med_l2,
            "snr_db_mean": float(np.mean(snrs)),
            "snr_db_std": float(np.std(snrs)),
            # interpretability robustness
            "spearman": spearman_ch(E_clean, E_adv),
            "jaccard_top5": jaccard_topk(E_clean, E_adv, k=5),
            "cosine_maps": cosine_maps(E_clean, E_adv),
            "norm_l2_shift": norm_l2_shift(E_clean, E_adv),
            "steps": steps,
            "alpha": float(alpha_rule(eps)) if steps is not None else None,
            "restarts": 1,
            "targeted": False,
            "auc": robust_auc(eps_grid, acc_list)
        }
        out["rows"].append(row)
    return out

# constructors for torchattacks
def FGSM_ctor(eps):                return ta.FGSM(model, eps=eps)
def BIM_ctor(eps, steps, alpha):   return ta.BIM(model, eps=eps, alpha=alpha, steps=steps)
def PGD_ctor(eps, steps, alpha):   return ta.PGD(model, eps=eps, alpha=alpha, steps=steps, random_start=True)
def MIM_ctor(eps, steps, alpha):   return ta.MIFGSM(model, eps=eps, alpha=alpha, steps=steps, decay=1.0)

eps_grid = [0.01, 0.02, 0.03, 0.05]
fgsm_res = run_iter_attack(FGSM_ctor, "FGSM", eps_grid, steps=None)
bim_res  = run_iter_attack(lambda e,s,a: BIM_ctor(e,s,a), "BIM", eps_grid, steps=10)
pgd_res  = run_iter_attack(lambda e,s,a: PGD_ctor(e,s,a), "PGD", eps_grid, steps=40)
mim_res  = run_iter_attack(lambda e,s,a: MIM_ctor(e,s,a), "MIM", eps_grid, steps=40)


## L₂‑style attacks (DeepFool, CW)

In [ ]:
# DeepFool (L2-ish minimal)
deepfool = ta.DeepFool(model, steps=50)
# CW-L2 (slow; consider a smaller subset)
cw = ta.CW(model, c=1e-3, steps=500, lr=0.01)

def run_point_attack(atk, name):
    B=128; preds_adv=[]; l2_succ=[]; snrs=[]
    X_adv_all=[]
    for i in range(0, X.size(0), B):
        Xi, yi = X[i:i+B], y[i:i+B]
        Xi = Xi.detach().clone().requires_grad_(True)
        x_adv = atk(Xi, yi).detach()
        x_adv = per_channel_clamp(x_adv, train_min_t, train_max_t)
        X_adv_all.append(x_adv)
        with torch.no_grad():
            pa = model(x_adv).argmax(1)
        preds_adv.append(pa.cpu())
        d = (x_adv - Xi).flatten(1); l2 = d.norm(p=2, dim=1).cpu().numpy()
        succ = (pa != yi).cpu().numpy()
        l2_succ.extend(l2[succ]); snrs.extend(snr_db(Xi, x_adv))
    preds_adv = torch.cat(preds_adv)
    adv_acc = float((preds_adv == y.cpu()).float().mean().item())
    X_adv = torch.cat(X_adv_all,0)
    E_adv = ig.attribute(X_adv, target=y, n_steps=50)
    return {
        "attack": name, "clean_acc": clean_acc, "adv_acc": adv_acc,
        "asr": 1-adv_acc, "median_L2_success": float(np.median(l2_succ)) if len(l2_succ) else float("nan"),
        "snr_db_mean": float(np.mean(snrs)), "snr_db_std": float(np.std(snrs)),
        "spearman": spearman_ch(E_clean, E_adv), "jaccard_top5": jaccard_topk(E_clean, E_adv, 5),
        "cosine_maps": cosine_maps(E_clean, E_adv), "norm_l2_shift": norm_l2_shift(E_clean, E_adv)
    }

deepfool_res = run_point_attack(deepfool, "DeepFool_L2")
# cw_res = run_point_attack(cw, "CW_L2")  # enable if you have time


## Masking sanity check (AutoAttack subset) + black‑box (Square, FAB)

In [ ]:
# AutoAttack (L∞) on a subset (e.g., first 512 samples)
aa_eps = 0.03
aa = AutoAttack(model, norm='Linf', eps=aa_eps, version='standard')
idx = slice(0, min(512, X.size(0)))
Xaa = X[idx]; yaa = y[idx]
Xaa_adv = aa.run_standard_evaluation(Xaa, yaa, bs=128)  # returns adv examples
Xaa_adv = per_channel_clamp(Xaa_adv, train_min_t, train_max_t)
with torch.no_grad():
    adv_acc_aa = float((model(Xaa_adv).argmax(1) == yaa).float().mean().item())
E_adv_aa = ig.attribute(Xaa_adv, target=yaa, n_steps=50)
aa_res = {
    "attack":"AutoAttack_std","eps": aa_eps, "subset_N": int(Xaa.size(0)),
    "adv_acc": adv_acc_aa, "asr": 1-adv_acc_aa,
    "spearman": spearman_ch(E_clean[idx], E_adv_aa),
    "jaccard_top5": jaccard_topk(E_clean[idx], E_adv_aa, 5),
    "cosine_maps": cosine_maps(E_clean[idx], E_adv_aa),
    "norm_l2_shift": norm_l2_shift(E_clean[idx], E_adv_aa)
}

# Foolbox (use scalar bounds!)
lower = float(np.min(train_min)); upper = float(np.max(train_max))
fmodel = fb.PyTorchModel(model, bounds=(lower, upper))
# Square (black-box)
raw, clipped, is_adv = fb.attacks.SquareAttack()(fmodel, Xaa.detach().cpu().numpy(), yaa.detach().cpu().numpy(), epsilons=aa_eps)
adv_acc_sq = float(1 - is_adv.mean())
sq_res = {"attack":"Square_Linf","eps":aa_eps,"subset_N":int(Xaa.size(0)),"adv_acc":adv_acc_sq,"asr":1-adv_acc_sq}
# FAB (white-box)
raw, clipped, is_adv = fb.attacks.FABAttack()(fmodel, Xaa.detach().cpu().numpy(), yaa.detach().cpu().numpy(), epsilons=aa_eps)
adv_acc_fab = float(1 - is_adv.mean())
fab_res = {"attack":"FAB_Linf","eps":aa_eps,"subset_N":int(Xaa.size(0)),"adv_acc":adv_acc_fab,"asr":1-adv_acc_fab}


# Save & pretty‑print

In [ ]:
single_run_report = {
    "clean_acc": clean_acc,
    "FGSM": fgsm_res,
    "BIM": bim_res,
    "PGD": pgd_res,
    "MIM": mim_res,
    "DeepFool_L2": deepfool_res,
    # "CW_L2": cw_res,
    "AutoAttack_std": aa_res,
    "Square_Linf": sq_res,
    "FAB_Linf": fab_res
}

from pprint import pprint
pprint(single_run_report)  # or:
open("subject1_seedX_single_run.json","w").write(json.dumps(single_run_report, indent=2))


## For TSNE and PCA illustrations

In [ ]:
with torch.no_grad():
    feats_clean = model.features(X)  # shape (N, D)
    feats_adv   = model.features(X_adv)
np.save(f"{base}/feats_clean_eps{eps}.npy", feats_clean.cpu().numpy())
np.save(f"{base}/feats_adv_eps{eps}.npy", feats_adv.cpu().numpy())


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pca = PCA(n_components=2).fit_transform(np.vstack([feats_clean, feats_adv]))
tsne = TSNE(n_components=2).fit_transform(np.vstack([feats_clean, feats_adv]))


# Single attack runner for PGD

In [32]:
import torchattacks as ta

def run_pgd_sweep(model, X, y, train_min_t, train_max_t, train_std, eps_grid, batch_size=128, steps=40):
    """
    model: nn.Module in eval()
    X,y: tensors on device, shapes (N,C,T), (N,)
    train_min_t, train_max_t: tensors broadcastable to (N,C,T)
    train_std: numpy (C,) or (1,C,1) from preprocessing.json
    """
    N = X.size(0)

    results = {
        "eps_grid": eps_grid,
        "acc_adv": [],
        "asr": [],
        "median_L2_success": [],
        "failure_rate": [],
        "auc": None,
        "per_eps": []  # detailed per-epsilon records
    }

    for eps in eps_grid:
        atk = ta.PGD(model, eps=eps, alpha=eps/8, steps=steps, random_start=True)
        preds_adv = []
        succ_mask_all = []
        l2_succ_all = []
        snr_all = []

        # craft adversarial set in batches (consistent order)
        for i in range(0, N, batch_size):
            Xi, yi = X[i:i+batch_size], y[i:i+batch_size]
            x_adv = atk(Xi, yi).detach()
            x_adv = per_channel_clamp(x_adv, train_min_t, train_max_t)

            with torch.no_grad():
                logits_adv = model(x_adv)
                pa = logits_adv.argmax(1)
            succ = (pa != yi)  # untargeted

            # metrics
            delta = (x_adv - Xi).view(Xi.size(0), -1)
            l2 = delta.norm(p=2, dim=1).cpu().numpy()
            l2_succ_all.extend(l2[succ.cpu().numpy()])
            snr_all.extend(snr_db(Xi, x_adv))

            preds_adv.append(pa.cpu())
            succ_mask_all.append(succ.cpu())

        preds_adv = torch.cat(preds_adv)            # (N,)
        succ_mask = torch.cat(succ_mask_all).numpy()  # (N,)

        acc = float((preds_adv == y.cpu()).float().mean().item())
        asr = 1.0 - acc
        med_l2 = float(np.median(l2_succ_all)) if len(l2_succ_all) else float("nan")
        fail_rate = float(1.0 - succ_mask.mean())    # fraction not fooled

        # store summaries
        results["acc_adv"].append(acc)
        results["asr"].append(asr)
        results["median_L2_success"].append(med_l2)
        results["failure_rate"].append(fail_rate)

        # store detail for this eps
        eps_uV = eps_to_uV_per_channel(eps, train_std).tolist()
        results["per_eps"].append({
            "eps": float(eps),
            "eps_uV_per_channel": eps_uV,
            "acc_adv": acc,
            "asr": asr,
            "median_L2_success": med_l2,
            "failure_rate": fail_rate,
            "snr_db_mean": float(np.mean(snr_all)),
            "snr_db_std": float(np.std(snr_all)),
            "steps": steps,
            "alpha": float(eps/8),
            "restarts": 1,
            "targeted": False
        })

    results["auc"] = robust_auc(eps_grid, results["acc_adv"])
    return results


In [ ]:


# 0) tensors and stats
train_set, test_set, train_subset, val_subset = load_subject_data_cached("BNCI2014001", subject_id)

X_train = np.stack([train_set[i][0] for i in range(len(train_set))])  # (N,C,T

X_test = np.stack([test_set[i][0] for i in range(len(test_set))])  # (N,C,T
y_test = np.array(test_set.get_metadata().target)                   # (N,)

train_mean = X_train.mean(axis=(0,2), keepdims=True)  # (1,C,1)
train_std  = X_train.std(axis=(0,2), keepdims=True) + 1e-6

# Optionally, keep empirical bounds for later clipping in attack
train_min = X_train.min(axis=(0,2), keepdims=True)
train_max = X_train.max(axis=(0,2), keepdims=True)

X = torch.tensor(X_test, dtype=torch.float32, device=device)  # (N,C,T)
y = torch.tensor(y_test, dtype=torch.long, device=device)

train_min_t = torch.tensor(train_min, dtype=torch.float32, device=device)  # (1,C,1)
train_max_t = torch.tensor(train_max, dtype=torch.float32, device=device)
train_std_np = np.array(train_std).reshape(-1)  # (C,)

# 1) eval mode
model.eval()

# 2) sweep
eps_grid = [0.01, 0.02, 0.03, 0.05]
pgd_summary = run_pgd_sweep(model, X, y, train_min_t, train_max_t, train_std_np, eps_grid, batch_size=128, steps=40)

import json
print(json.dumps(pgd_summary, indent=2))

# 3) save
base = "explainability"
with open(f"{base}/attack_metadata.json", "w") as f:
    json.dump({"PGD_Linf": pgd_summary}, f, indent=2)


In [17]:
# FGSM / BIM / PGD
eps = 0.05
import torchattacks as ta
fgsm = ta.FGSM(model, eps=eps)
bim  = ta.BIM(model, eps=eps, alpha=eps/10, steps=10)
pgd  = ta.PGD(model, eps=eps, alpha=eps/8, steps=40, random_start=True)

# DeepFool / CW
deepfool = ta.DeepFool(model, steps=50)
cw = ta.CW(model, c=1e-3, steps=1000, lr=0.01)

# AutoAttack (subset for masking check)
from autoattack import AutoAttack
aa = AutoAttack(model, norm='Linf', eps=eps, version='standard')



setting parameters for standard version
